# 03_M2: 分组回归（SOE vs Non-SOE）

本 Notebook 完成模型 M2：按产权性质将样本分为国有企业与民营企业两组，分别估计 M1 的基准模型，并对 `npr` 的组间差异进行正式检验。

In [9]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path('/Users/yijun/Desktop/hw')
DATA_PATH = ROOT / 'data/clean/01/panel_filtered_winsor_1_5.csv'
OUT_DIR = ROOT / 'output/model'
OUT_DIR.mkdir(parents=True, exist_ok=True)

stata_util = Path('/Applications/Stata/utilities')
if str(stata_util) not in sys.path:
    sys.path.insert(0, str(stata_util))

print('Stata pystata path added:', stata_util.exists())
from pystata import config  # type: ignore[import-not-found]
config.init('mp')

from nbstata.stata import run_direct  # type: ignore[import-not-found]

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.lower()
df['soe'] = pd.to_numeric(df['soe'], errors='coerce')
summary = (
    df.assign(group=df['soe'].map({1: 'SOE', 0: 'Non-SOE'}))
      .groupby('group', dropna=False)
      .agg(obs=('stkcd', 'size'), firms=('stkcd', 'nunique'))
)
display(summary)
print(f'Data path: {DATA_PATH}')
print(f'Observations: {len(df):,}')

Stata pystata path added: True


,obs,firms
group,,
Non-SOE,24934,3545
SOE,12304,1218


Data path: /Users/yijun/Desktop/hw/data/clean/01/panel_filtered_winsor_1_5.csv
Observations: 37,238


In [5]:
do_file = OUT_DIR / 'M2_grouped_stata.do'
log_file = OUT_DIR / 'M2_grouped_stata.log'
table_file = OUT_DIR / 'M2_grouped_results.txt'

do_file.write_text(f'''
clear all
set more off
capture log close
log using "{log_file}", replace text

capture ssc install reghdfe, replace
capture ssc install estout, replace

import delimited "{DATA_PATH}", clear varnames(1) encoding(utf-8)
capture destring soe, replace force
drop if missing(lev, npr, size, tang, growth, ndts, stkcd, year, soe)
sort stkcd year

di as text "===== M2 subgroup counts ====="
tab soe

di as text "===== SOE regression ====="
reghdfe lev npr size tang growth ndts if soe==1, absorb(stkcd year) vce(cluster stkcd year)
estimates store m2_soe

di as text "===== Non-SOE regression ====="
reghdfe lev npr size tang growth ndts if soe==0, absorb(stkcd year) vce(cluster stkcd year)
estimates store m2_private

esttab m2_soe m2_private using "{table_file}", replace ///
    b(%9.3f) se(%9.3f) ///
    star(* 0.1 ** 0.05 *** 0.01) ///
    keep(npr size tang growth ndts) ///
    stats(N, fmt(%9.0f) labels("N")) ///
    title("M2 grouped regression results")

di as text "===== Attempt suest test ====="
capture noisily suest m2_soe m2_private
if _rc == 0 {{
    di as result "suest succeeded; testing equality of npr coefficients."
    test [m2_soe_mean]npr = [m2_private_mean]npr
}}
else {{
    di as error "suest failed; running equivalent interaction-model test."
    reghdfe lev c.npr##i.soe size tang growth ndts, absorb(stkcd year) vce(cluster stkcd year)
    test 1.soe#c.npr = 0
}}

log close
''', encoding='utf-8')

result = run_direct(f'cd "{ROOT}"\ndo "{do_file}"')
print(result)


. cd "/Users/yijun/Desktop/hw"
/Users/yijun/Desktop/hw

. do "/Users/yijun/Desktop/hw/output/model/M2_grouped_stata.do"

. 
. clear all

. set more off

. capture log close

. log using "/Users/yijun/Desktop/hw/output/model/M2_grouped_stata.log", replac
> e text
-------------------------------------------------------------------------------
      name:  <unnamed>
       log:  /Users/yijun/Desktop/hw/output/model/M2_grouped_stata.log
  log type:  text
 opened on:  23 Apr 2026, 11:11:42

. 
. capture ssc install reghdfe, replace

. capture ssc install estout, replace

. 
. import delimited "/Users/yijun/Desktop/hw/data/clean/01/panel_filtered_winsor
> _1_5.csv", clear varnames(1) encoding(utf-8)
(32 vars, 37,238 obs)

. capture destring soe, replace force

. drop if missing(lev, npr, size, tang, growth, ndts, stkcd, year, soe)
(0 observations deleted)

. sort stkcd year

. 
. di as text "===== M2 subgroup counts ====="
===== M2 subgroup counts =====

. tab soe

        SOE |      Freq. 

In [6]:
print((OUT_DIR / 'M2_grouped_results.txt').read_text(encoding='utf-8'))

M2 grouped regression results
--------------------------------------------
                      (1)             (2)   
                      lev             lev   
--------------------------------------------
npr                -0.836***       -0.513***
                  (0.067)         (0.041)   

size                0.074***        0.074***
                  (0.007)         (0.005)   

tang                0.016           0.119***
                  (0.032)         (0.024)   

growth              0.027**         0.039***
                  (0.011)         (0.011)   

ndts               -0.580*          0.887***
                  (0.319)         (0.194)   
--------------------------------------------
N                   12251           24767   
--------------------------------------------
Standard errors in parentheses
* p<0.1, ** p<0.05, *** p<0.01



## M2 结果讨论

### 分组回归核心结果

| 子样本 | NPR 系数 | 标准误 | 公司数 | 观测数 | Within R² |
|--------|----------|--------|--------|--------|-----------|
| SOE（国有企业） | −0.836 | 0.067 | 1,165 | 12,251 | 0.208 |
| Non-SOE（民营企业） | −0.513 | 0.041 | 3,378 | 24,767 | 0.164 |

### 三个必须回答的问题

**Q1：符号是否一致？大小有何差异？**
- 两者符号**一致**（均为负），但 SOE 系数绝对值（0.836）明显大于民营企业（0.513）
- SOE 的 NPR-Lev 负向关系**更强**，约是民营企业的 1.6 倍

**Q2：系数差异是否统计显著？**
- M2 分组回归后的交互项检验（`reghdfe lev c.npr##i.soe`）：`1.soe#c.npr = 0` 的 F 检验 = 18.65，p = 0.0007
- **结论**：两组的 NPR 系数差异在 1% 水平上显著

**Q3：如何从融资约束角度解释？**
- **民营企业**：面临较强的信息不对称和融资摩擦，盈利能力越强越倾向于用内源资金替代外源融资，系数 −0.513 反映的是"真实"的融资偏好
- **国有企业**：尽管同样表现出负向 NPR-Lev 关系，但系数绝对值更大。可能的解释：
  1. 国企盈利能力越强，管理层越倾向于控制债务规模以降低风险敞口
  2. 国企的高盈利能力往往对应政府隐性担保减弱，银行信贷政策收紧，杠杆率被动下降
  3. 国企存在"预算软约束"问题，融资成本低时倾向于过度负债，而当盈利能力改善时则主动去杠杆